# Predicting Residential EV Charging Loads using Neural Networks
### 📓 My project notes & reference (Codecademy)

A record of how I built this project, with my code, outputs, and review notes for each step.

**Big picture:** clean per-session EV charging data → add nearby traffic counts → fit a linear
regression baseline → train a PyTorch neural network → compare. The network's nonlinearity beat
the linear baseline.

**Final scores (test-set MSE, lower = better):**

| Model | Test MSE | ≈ avg miss (√MSE) |
|---|---|---|
| Linear Regression (baseline) | 131.42 | ~11.5 kWh |
| Neural Net — 3000 epochs | 122.73 | ~11.1 kWh |
| Neural Net — 4500 epochs (`model4500.pth`) | 115.22 | ~10.7 kWh |

> **Note — data was already one-hot encoded.** Codecademy's `EV charging reports.csv` already has
> the categoricals expanded into 0/1 columns (`User_private`, `month_plugin_*`,
> `weekdays_plugin_*`). That's why the column count is so high (32 → 39 after merge) and why the
> final 26-feature `model4500.pth` loads against my data without extra encoding.

In [ ]:
# Setup - import basic data libraries
import numpy as np
import pandas as pd

## Task Group 1 — Load, Inspect, and Merge Datasets

### Task 1 — Load the EV charging data
Import `datasets/EV charging reports.csv` into `ev_charging_reports` and preview with `.head()`.
Per-session data: user/garage info, plug-in/out times, charging loads (`El_kWh`), session dates.

In [ ]:
ev_charging_reports = pd.read_csv("datasets/EV charging reports.csv")
ev_charging_reports.head()

[5 rows x 32 columns]  (session_ID, Garage_ID, User_ID, User_private, Shared_ID,
  Start_plugin, Start_plugin_hour, End_plugout, End_plugout_hour, El_kWh, ... ,
  month_plugin_* (one-hot), weekdays_plugin_* (one-hot))

📝 **Notes**
- `pd.read_csv(path)` → loads a CSV into a DataFrame. No `sep` needed here (Codecademy's file is
  comma-delimited).
- `.head()` → first 5 rows, quick sanity check.
- Already **32 columns** because the month/weekday categoricals are pre-one-hot-encoded.
- `El_kWh` is the **target** I'll predict. It shows up as text (`0,3`) — European comma decimals
  to fix later.

### Task 2 — Load the local traffic data
Import `datasets/Local traffic distribution.csv` into `traffic_reports`. Hourly traffic-density
counts at 5 nearby locations.

In [ ]:
traffic_reports = pd.read_csv("datasets/Local traffic distribution.csv")
traffic_reports.head()

   Date_from           Date_to            Kroppan_bru  Moholtlia  Selsbakk  Moholt_rampe_2  Jonsvannsveien...
0  01.12.2018 00:00    01.12.2018 01:00         639          0         0            4            144
1  01.12.2018 01:00    01.12.2018 02:00         487        153       115           21             83
2  01.12.2018 02:00    01.12.2018 03:00         408         85        75           10             69
3  01.12.2018 03:00    01.12.2018 04:00         282         89        56            8             39
4  01.12.2018 04:00    01.12.2018 05:00         165         64        34            3             25

📝 **Notes**
- Key columns: `Date_from` / `Date_to` (the hour window) + one count column per sensor.
- `Date_from` is the column I'll join to the charging data on.

### Task 3 — Merge charging + traffic
Merge into `ev_charging_traffic`, joining `Start_plugin_hour` (charging) to `Date_from` (traffic).

In [ ]:
ev_charging_traffic = ev_charging_reports.merge(traffic_reports, left_on='Start_plugin_hour', right_on='Date_from')
ev_charging_traffic.head()

[5 rows x 39 columns]   (32 charging cols + 7 traffic cols, matched on the plug-in hour)

📝 **Notes**
- `df1.merge(df2, left_on='A', right_on='B')` → SQL-style join where `df1.A == df2.B`.
- Each charging session gets the traffic counts for the hour it started.
- 32 + 7 = **39 columns** after the merge.

### Task 4 — Inspect dtypes & missing values
`.info()` to see data types and non-null counts.

In [ ]:
ev_charging_traffic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6833 entries, 0 to 6832
Data columns (total 39 columns):
  ... El_kWh            object   (should be numeric!)
  ... Duration_hours    object   (should be numeric!)
  ... Shared_ID         1399 non-null  (lots of missing -> will drop)
  ... month_plugin_* / weekdays_plugin_*  float64 (already one-hot)
  ... traffic columns   int64
dtypes: float64(21), int64(6), object(12)
memory usage: 2.0+ MB

📝 **Notes**
- `.info()` → column names, **non-null counts** (missing data), and **dtypes**.
- Red flags: `El_kWh` and `Duration_hours` are `object` (text) because of comma decimals;
  `Shared_ID` is mostly missing (1399 / 6833).
- These get cleaned/dropped in Task Group 2.

## Task Group 2 — Data Cleaning and Preparation

### Task 5 — Drop columns not used for training
Drop ID columns, mostly-missing columns, and leftover non-numeric columns.

In [ ]:
drop_columns = ['session_ID', 'Garage_ID', 'User_ID',
                'Shared_ID',
                'Plugin_category','Duration_category',
                'Start_plugin', 'Start_plugin_hour', 'End_plugout', 'End_plugout_hour',
                'Date_from', 'Date_to']

ev_charging_traffic = ev_charging_traffic.drop(columns=drop_columns, axis=1)
ev_charging_traffic.head()

[5 rows x 27 columns]   (39 - 12 dropped = 27)

📝 **Notes**
- `df.drop(columns=cols)` → returns the frame without those columns.
- `axis=1` = columns (`axis=0` = rows). It's **redundant** here since I used `columns=`, but
  harmless.
- 39 → **27 columns** left.

### Task 6 — Fix European decimals (`,` → `.`)
`El_kWh` and `Duration_hours` are text because commas are used as decimal points. Replace `,`
with `.` in the text columns.

In [ ]:
for column in ev_charging_traffic.columns:
    if ev_charging_traffic[column].dtype == 'object':
        ev_charging_traffic[column] = ev_charging_traffic[column].str.replace(',', '.')

ev_charging_traffic.head()

El_kWh and Duration_hours now read like 0.3 / 0.05 (period decimals) — still object dtype, converted to float in Task 7.

📝 **Notes**
- Loop over columns; `df[col].dtype == 'object'` finds the **text** columns.
  (Use `.dtype` — `df[col]` alone is the whole Series and `if Series == 'object'` raises
  *"truth value of a Series is ambiguous"*.)
- `df[col].str.replace(',', '.')` → element-wise string replace via the `.str` accessor.
- This only swaps the character; the column is still `object` until the cast in Task 7.

### Task 7 — Convert all columns to float
Cast every column to `float` so the data is fully numeric for modeling.

In [ ]:
for column in ev_charging_traffic.columns:
    ev_charging_traffic[column] = ev_charging_traffic[column].astype(float)

ev_charging_traffic.head()

[5 rows x 27 columns]  — every column now float64 (e.g. 0.30, 0.136667, 3244.0)

📝 **Notes**
- `df[col].astype(float)` → converts a column's dtype to float.
- Works here because Task 6 already made the numbers parseable and the categoricals were already
  numeric 0/1. Models need numeric input.

## Task Group 3 — Train / Test Split
Split into training data (fit the model) and testing data (evaluate it).

### Task 8 — Build X (features) and y (target)
`X` = all input features, `y` = the target `El_kWh`.

In [ ]:
numerical_features = ev_charging_traffic.drop(['El_kWh'], axis=1).columns
X = ev_charging_traffic[numerical_features]

y = ev_charging_traffic['El_kWh']

📝 **Notes**
- `drop(['El_kWh'], axis=1).columns` → every column name **except** the target → use it to select
  `X`.
- `X` = 26 feature columns, `y` = the single `El_kWh` column.
- Convention: capital `X` (matrix of features), lowercase `y` (target vector).

### Task 9 — 80/20 split with sklearn
Split `X`/`y` into train/test, 80% training, `random_state=2` for reproducibility.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    train_size=0.80,
                                                    test_size=0.20,
                                                    random_state=2) # set a random seed - do not modify

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

Training size: (5466, 26)
Testing size: (1367, 26)

📝 **Notes**
- `train_test_split(X, y, train_size=0.8, random_state=2)` → 4 arrays: `X_train/X_test/y_train/y_test`.
- `random_state` fixes the shuffle so I get the **same split every run** (reproducible results).
- Result: 5466 train rows, 1367 test rows, **26 features**.

## Task Group 4 — Linear Regression Baseline
Optional but useful: if a plain linear model does as well, no need for a neural net. It's the number to beat.

### Task 10 — Train the baseline
Fit a scikit-learn `LinearRegression` on the training data.

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

LinearRegression()

📝 **Notes**
- `LinearRegression()` then `.fit(X_train, y_train)` → learns the best straight-line weights.
- sklearn pattern: **create model → `.fit()` → `.predict()`**.

### Task 11 — Evaluate the baseline (MSE)
Compute test-set MSE with `mean_squared_error`, save to `test_mse`.

In [ ]:
from sklearn.metrics import mean_squared_error

linear_test_predictions = linear_model.predict(X_test)
test_mse = mean_squared_error(y_test, linear_test_predictions)
print("Linear Regression - Test Set MSE:", test_mse)

Linear Regression - Test Set MSE: 131.4188163356643

📝 **Notes**
- `mean_squared_error(y_true, y_pred)` → average squared difference.
- **MSE ≈ 131.4.** It's *squared* error, so √131.4 ≈ **11.5 kWh** = the model's average miss.
- This is the baseline the neural net needs to beat.

## Task Group 5 — Train a Neural Network in PyTorch

### Task 12 — Import PyTorch
`torch` (core), `nn` (layers + loss functions), `optim` (optimizers).

In [ ]:
import torch
from torch import nn
from torch import optim

### Task 13 — Convert data to tensors
PyTorch works on tensors, not DataFrames. Cast to `float`; reshape `y` to a column vector.

In [ ]:
# Convert training set
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float).view(-1,1)

# Convert testing set
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float).view(-1,1)

📝 **Notes**
- `torch.tensor(df.values, dtype=torch.float)` → DataFrame → tensor of floats.
- `.view(-1, 1)` → reshapes `y` from a flat row into a **column vector** so its shape matches the
  network's 1-node output. `-1` = "infer this dimension" (= number of rows).

### Task 14 — Build the network
Seed for reproducibility, then `nn.Sequential`: 26 → 56 (ReLU) → 26 (ReLU) → 1.

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(26,56),
    nn.ReLU(),
    nn.Linear(56, 26),
    nn.ReLU(),
    nn.Linear(26,1)
)

📝 **Notes**
- `nn.Sequential(...)` → stacks layers in order; data flows top to bottom.
- `nn.Linear(in, out)` → a fully-connected layer. Input **26** must match the feature count;
  output **1** is the single predicted `El_kWh`.
- `nn.ReLU()` → activation that adds **nonlinearity** (without it, the whole stack collapses to a
  linear model). This is what lets it beat the baseline.
- `torch.manual_seed(42)` → same random starting weights every run.

### Task 15 — Loss & optimizer
MSE loss; Adam optimizer at learning rate 0.0007.

In [ ]:
loss = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0007)

📝 **Notes**
- `nn.MSELoss()` → same metric as the baseline, so the comparison is fair.
- `optim.Adam(model.parameters(), lr=0.0007)` → Adam adjusts the weights; `lr` (learning rate) =
  step size. Too big overshoots, too small trains slowly.

### Task 16 — Training loop (3000 epochs)
Forward pass → compute loss → backprop → update weights. Print MSE every 500 epochs.

In [ ]:
num_epochs = 3000
for epoch in range(num_epochs):
    predictions = model(X_train_tensor)          # forward pass
    MSE = loss(predictions, y_train_tensor)      # compute loss
    MSE.backward()                  # compute gradients
    optimizer.step()                # update weights & biases
    optimizer.zero_grad()           # reset gradients for the next iteration

    if (epoch + 1) % 500 == 0:      # print every 500 epochs
        print(f'Epoch [{epoch + 1}/{num_epochs}], MSE Loss: {MSE.item():.2f}')

Epoch [500/3000], MSE Loss: 146.76
Epoch [1000/3000], MSE Loss: 139.67
Epoch [1500/3000], MSE Loss: 118.95
Epoch [2000/3000], MSE Loss: 116.22
Epoch [2500/3000], MSE Loss: 110.68
Epoch [3000/3000], MSE Loss: 107.03

📝 **Notes — the 4 lines that ARE training (memorize this loop):**
1. `predictions = model(X_train_tensor)` — **forward pass**: run inputs through the net.
2. `MSE = loss(predictions, y_train_tensor)` — measure how wrong it is.
3. `MSE.backward()` — **backprop**: compute gradients (which way each weight should move).
4. `optimizer.step()` — nudge the weights; then `optimizer.zero_grad()` to clear gradients.

⚠️ `zero_grad()` matters — PyTorch **accumulates** gradients, so without it they'd pile up across
epochs. Training MSE fell 146.76 → **107.03** = it's learning.

### Task 17 — Save the model
Persist the trained network to `models/model.pth`.

In [ ]:
# save the neural network
torch.save(model, 'models/model.pth')

📝 **Notes**
- `torch.save(model, path)` → saves the model so I can reload it later without retraining.

### Task 18 — Evaluate on the test set
Switch to eval mode, no gradients, compute test MSE.

In [ ]:
model.eval() # set the model to evaluation mode
with torch.no_grad(): # disable gradient calculations
    predictions = model(X_test_tensor)
    test_loss = loss(predictions, y_test_tensor)

print('Neural Network - Test Set MSE:', test_loss.item())

Neural Network - Test Set MSE: 122.73201751708984

📝 **Notes**
- `model.eval()` + `with torch.no_grad():` → evaluation mode, no gradient tracking (faster, and
  correct — I'm not training here).
- `.item()` → pulls the plain Python number out of a 1-element tensor.
- **Test MSE 122.73 < baseline 131.42** → the network already beats linear regression. ✅

### Task 19 — Load & evaluate the longer-trained model
`model4500.pth` was trained for 4500 epochs. Load and evaluate it.

In [ ]:
# load the model
model4500 = torch.load('models/model4500.pth')

model4500.eval()
with torch.no_grad():
    predictions = model4500(X_test_tensor)
    test_loss = loss(predictions, y_test_tensor)

print('Neural Network - Test Set MSE:', test_loss.item())

Neural Network - Test Set MSE: 115.21600341796875

📝 **Notes**
- `torch.load(path)` → restores a saved model.
- It loads cleanly because this dataset has **26 features**, matching what `model4500` expects
  (the categoricals were already one-hot encoded).
- **Test MSE 115.22** — a ~12% improvement over the linear baseline. More epochs → lower loss
  here. The nonlinearity (ReLU) is what made the net able to beat the straight-line model.

## Wrap-up & takeaways

**End-to-end pipeline I built:**
`read_csv → merge → drop cols → fix decimals → astype(float) → X/y → train_test_split →`
`LinearRegression (baseline) → tensors → nn.Sequential → MSELoss + Adam → train loop → evaluate`

**Results:** Linear 131.4 → NN(3000) 122.7 → NN(4500) 115.2. The neural network beat the baseline
because ReLU adds nonlinearity a straight line can't capture.

**Things to remember:**
- The 4-line training loop (forward → loss → `backward()` → `step()` + `zero_grad()`).
- `.view(-1, 1)` to shape the target to the output.
- Input layer size **must** equal the feature count (26 here).
- MSE is squared — take √ for the average miss in real units (kWh).

**Ideas to push further:** different feature sets, more/fewer hidden nodes, other activations,
different learning rates, more epochs.